# Modul 12: Unüberwachtes Lernen und Text als Merkmale | Lösungen

## Überblick

Sie vergleichen Clusterverfahren, bewerten Cluster vorsichtig, erkennen Anomalien und nutzen wenige Labels mit semi-supervised Lernen. Danach bereiten Sie eine kleine deutsche Textsammlung auf, erzeugen Sparse-Merkmale und trainieren reproduzierbare Textklassifikationspipelines.

**Zugehörige Vorlesungen**

- **Unüberwacht lernen**
- **Text als Merkmale**

## Lernziele

Nach der Bearbeitung können Sie:

- KMeans, MiniBatchKMeans, hierarchisches Clustering, DBSCAN und GaussianMixture passend zu Datenformen vergleichen.
- Anomalieerkennung und LabelSpreading auf skalierten Daten anwenden und Unsicherheit dokumentieren.
- Textdaten bereinigen, mit Count- und TF-IDF-Merkmalen darstellen und mit linearen Modellen klassifizieren.

## Geprüfte Fähigkeiten

- Clustering, Silhouette, Adjusted Rand Index und probabilistische Zuordnungen
- IsolationForest und semi-supervised LabelSpreading
- CountVectorizer, TfidfVectorizer, Sparse-Matrizen und Textpipelines

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** fortgeschritten
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle erzeugt zwei kleine geometrische Datensätze und eine lokale deutsche Textsammlung mit drei Klassen. Es sind keine externen Downloads oder Dateien erforderlich.

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

X_blobs, y_blobs = make_blobs(
    n_samples=360,
    centers=[(-4, -1), (0, 3), (4, -1)],
    cluster_std=[0.75, 1.0, 0.8],
    random_state=RANDOM_SEED,
)
X_moons, y_moons = make_moons(n_samples=360, noise=0.08, random_state=RANDOM_SEED)

texte = [
    "Das Team gewann nach einer starken zweiten Halbzeit.",
    "Der Trainer lobte die Abwehr und den schnellen Angriff.",
    "Das Fußballspiel endete nach Verlängerung unentschieden.",
    "Die Läuferin stellte beim Wettkampf einen neuen Rekord auf.",
    "Der Torwart hielt den entscheidenden Elfmeter.",
    "Im Finale überzeugte die Mannschaft mit ruhigem Passspiel.",
    "Ein neues Softwareupdate verbessert die Datensicherheit.",
    "Der Prozessor arbeitet schneller und benötigt weniger Energie.",
    "Forschende testen einen kompakten Roboter für die Produktion.",
    "Die App verarbeitet Bilder nun direkt auf dem Smartphone.",
    "Das Netzwerk erkennt ungewöhnliche Zugriffe automatisch.",
    "Die neue Python-Bibliothek vereinfacht die Datenanalyse.",
    "Die Stadt eröffnet einen neuen Park mit vielen Bäumen.",
    "Der Zugverkehr wird wegen Bauarbeiten am Wochenende geändert.",
    "Die Schule plant zusätzliche Kurse für das kommende Jahr.",
    "Im Krankenhaus wurde eine neue Ambulanz eröffnet.",
    "Der Gemeinderat diskutiert günstigere Wohnungen.",
    "Die Bibliothek verlängert ihre Öffnungszeiten im Sommer.",
] * 3
labels = (["Sport"] * 6 + ["Technik"] * 6 + ["Alltag"] * 6) * 3

text_daten = pd.DataFrame({"Text": texte, "Kategorie": labels})
# Kleine Variationen verhindern vollständig identische Duplikate.
text_daten["Text"] = [f"{text} Hinweis {i % 3}." for i, text in enumerate(text_daten["Text"])]

print("Blob-Daten:", X_blobs.shape)
print("Moon-Daten:", X_moons.shape)
print("Textdaten:", text_daten.shape)

### Aufgabe 1: KMeans, MiniBatch und hierarchisches Clustering vergleichen

Standardisieren Sie `X_blobs` und passen Sie KMeans, MiniBatchKMeans und AgglomerativeClustering jeweils mit drei Clustern an. Berechnen Sie für jede Zuordnung Silhouette und Adjusted Rand Index gegenüber den nur zur nachträglichen Kontrolle bereitstehenden wahren Labels `y_blobs`.

Visualisieren Sie die drei Zuordnungen. Verwenden Sie die wahren Labels nicht beim Anpassen der Clusterverfahren.

In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score

# ============================================================
# MUSTERLÖSUNG
# ============================================================

scaler_cluster = StandardScaler()
X_blobs_skaliert = scaler_cluster.fit_transform(X_blobs)

cluster_modelle = {
    "KMeans": KMeans(n_clusters=3, n_init=10, random_state=RANDOM_SEED),
    "MiniBatchKMeans": MiniBatchKMeans(
        n_clusters=3,
        batch_size=64,
        n_init=10,
        random_state=RANDOM_SEED,
    ),
    "Agglomerativ": AgglomerativeClustering(n_clusters=3),
}

zuordnungen = {}
berichte = []
for name, cluster_modell in cluster_modelle.items():
    # fit_predict() lernt ausschließlich aus den Merkmalen und gibt Clusterindizes zurück.
    cluster_labels = cluster_modell.fit_predict(X_blobs_skaliert)
    zuordnungen[name] = cluster_labels
    berichte.append(
        {
            "Verfahren": name,
            "Silhouette": silhouette_score(X_blobs_skaliert, cluster_labels),
            "ARI_nur_zur_Kontrolle": adjusted_rand_score(y_blobs, cluster_labels),
        }
    )

display(pd.DataFrame(berichte).round(3))

for name, cluster_labels in zuordnungen.items():
    plt.figure()
    plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=cluster_labels, s=20)
    plt.xlabel("Merkmal 1")
    plt.ylabel("Merkmal 2")
    plt.title(name)
    plt.show()

> **Musterantwort und Interpretation**
>
> Der Adjusted Rand Index vergleicht eine Clusterzuordnung mit bekannten Referenzklassen. In einer echten unüberwachten Aufgabe fehlen solche Labels gerade deshalb, weil Strukturen erst entdeckt werden sollen. Dann müssen interne Kennzahlen wie Silhouette, Stabilitätsprüfungen, Visualisierungen und vor allem fachliche Plausibilität gemeinsam betrachtet werden.

### Aufgabe 2: Komplexe Formen mit DBSCAN und GaussianMixture untersuchen

Standardisieren Sie `X_moons`. Passen Sie KMeans mit zwei Clustern, DBSCAN und ein GaussianMixture-Modell mit zwei Komponenten an. Visualisieren Sie die Zuordnungen und geben Sie für DBSCAN die Anzahl erkannter Cluster und Rauschpunkte aus.

Lassen Sie das GaussianMixture-Modell zusätzlich für die ersten fünf Punkte Komponentenwahrscheinlichkeiten ausgeben.

In [ ]:
from sklearn.cluster import DBSCAN, KMeans
from sklearn.mixture import GaussianMixture

# ============================================================
# MUSTERLÖSUNG
# ============================================================

X_moons_skaliert = StandardScaler().fit_transform(X_moons)

kmeans_moons = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_SEED)
dbscan_moons = DBSCAN(eps=0.28, min_samples=6)
gmm_moons = GaussianMixture(n_components=2, covariance_type="full", random_state=RANDOM_SEED)

labels_kmeans = kmeans_moons.fit_predict(X_moons_skaliert)
labels_dbscan = dbscan_moons.fit_predict(X_moons_skaliert)
labels_gmm = gmm_moons.fit_predict(X_moons_skaliert)

for name, labels_aktuell in {
    "KMeans": labels_kmeans,
    "DBSCAN": labels_dbscan,
    "GaussianMixture": labels_gmm,
}.items():
    plt.figure()
    plt.scatter(X_moons[:, 0], X_moons[:, 1], c=labels_aktuell, s=20)
    plt.title(name)
    plt.xlabel("Merkmal 1")
    plt.ylabel("Merkmal 2")
    plt.show()

# DBSCAN verwendet -1 für Punkte, die keinem dichten Cluster zugeordnet werden.
cluster_ohne_rauschen = set(labels_dbscan) - {-1}
print("DBSCAN-Cluster:", len(cluster_ohne_rauschen))
print("DBSCAN-Rauschpunkte:", int(np.sum(labels_dbscan == -1)))

wahrscheinlichkeiten = gmm_moons.predict_proba(X_moons_skaliert[:5])
print("GMM-Komponentenwahrscheinlichkeiten für fünf Punkte:")
print(np.round(wahrscheinlichkeiten, 3))
print("Zeilensummen:", np.round(wahrscheinlichkeiten.sum(axis=1), 6))

> **Musterantwort und Interpretation**
>
> KMeans ordnet Punkte dem jeweils nächsten Zentrum zu und bevorzugt dadurch kompakte, ungefähr kugelförmige Bereiche. Zwei ineinander liegende Halbmonde lassen sich durch solche Voronoi-Bereiche schlecht darstellen. DBSCAN kann zusammenhängende dichte Formen verfolgen, reagiert dafür aber empfindlich auf Skalierung sowie die Wahl von eps und min_samples.

### Aufgabe 3: Anomalien erkennen und fachlich prüfen

Erweitern Sie `X_blobs` um acht weit entfernte künstliche Punkte. Skalieren Sie die erweiterten Daten und passen Sie einen IsolationForest mit einem erwarteten Anomalieanteil von ungefähr zwei Prozent an.

Geben Sie die erkannten Anomalieindizes und deren Werte aus. Visualisieren Sie normale und auffällige Punkte und prüfen Sie, wie viele der acht künstlichen Punkte erkannt wurden.

In [ ]:
from sklearn.ensemble import IsolationForest

# ============================================================
# MUSTERLÖSUNG
# ============================================================

kuenstliche_anomalien = np.array(
    [[-8, -7], [-7, 7], [8, 7], [9, -5], [-9, 3], [7, 9], [0, -8], [10, 1]],
    dtype=float,
)
X_mit_anomalien = np.vstack([X_blobs, kuenstliche_anomalien])
X_anomalien_skaliert = StandardScaler().fit_transform(X_mit_anomalien)

anomalie_modell = IsolationForest(
    n_estimators=150,
    contamination=0.02,
    random_state=RANDOM_SEED,
)
# IsolationForest kennzeichnet Anomalien mit -1 und normale Punkte mit 1.
anomalie_label = anomalie_modell.fit_predict(X_anomalien_skaliert)
anomalie_indizes = np.flatnonzero(anomalie_label == -1)

print("Erkannte Anomalieindizes:", anomalie_indizes.tolist())
print("Erkannte Werte:")
print(np.round(X_mit_anomalien[anomalie_indizes], 2))

erste_kuenstliche_position = len(X_blobs)
erkannte_kuenstliche = np.sum(anomalie_indizes >= erste_kuenstliche_position)
print(f"Erkannte künstliche Anomalien: {erkannte_kuenstliche} von {len(kuenstliche_anomalien)}")

plt.scatter(
    X_mit_anomalien[anomalie_label == 1, 0],
    X_mit_anomalien[anomalie_label == 1, 1],
    s=18,
    label="normal",
)
plt.scatter(
    X_mit_anomalien[anomalie_label == -1, 0],
    X_mit_anomalien[anomalie_label == -1, 1],
    s=70,
    marker="x",
    label="auffällig",
)
plt.title("IsolationForest-Ergebnis")
plt.legend()
plt.show()

> **Musterantwort und Interpretation**
>
> Das Verfahren erkennt Punkte, die statistisch ungewöhnlich isoliert liegen. Ein solcher Punkt kann ein Messfehler, ein seltener aber gültiger Zustand oder ein wichtiges Ereignis sein. Erst Herkunft, Messprozess, Zeitkontext und Fachwissen erlauben eine belastbare Einordnung.

### Aufgabe 4: Mit wenigen Labels lernen

Verwenden Sie die standardisierten Blob-Daten. Lassen Sie pro wahrer Klasse nur drei zufällig ausgewählte Labels sichtbar und setzen Sie alle anderen Zielwerte auf `-1`. Trainieren Sie `LabelSpreading` und berechnen Sie die Genauigkeit der übertragenen Labels ausschließlich auf den zuvor unbeschrifteten Punkten.

Geben Sie außerdem die fünf unbeschrifteten Punkte mit der höchsten Unsicherheit aus. Nutzen Sie dazu die größte Klassenwahrscheinlichkeit als Sicherheitsmaß.

In [ ]:
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import accuracy_score

# ============================================================
# MUSTERLÖSUNG
# ============================================================

y_wenig_labels = np.full_like(y_blobs, fill_value=-1)

# Für jede Klasse bleiben nur drei zufällig ausgewählte Beispiele beschriftet.
for klasse in np.unique(y_blobs):
    klassen_indizes = np.flatnonzero(y_blobs == klasse)
    sichtbare_indizes = rng.choice(klassen_indizes, size=3, replace=False)
    y_wenig_labels[sichtbare_indizes] = klasse

semi_modell = LabelSpreading(kernel="rbf", gamma=10, alpha=0.2, max_iter=100)
semi_modell.fit(X_blobs_skaliert, y_wenig_labels)

unbeschriftet = y_wenig_labels == -1
uebertragene_labels = semi_modell.transduction_[unbeschriftet]
print(
    "Genauigkeit auf zuvor unbeschrifteten Punkten:",
    round(accuracy_score(y_blobs[unbeschriftet], uebertragene_labels), 3),
)

# label_distributions_ enthält für jeden Punkt eine Klassenverteilung.
sicherheit = semi_modell.label_distributions_[unbeschriftet].max(axis=1)
unsicherste_lokal = np.argsort(sicherheit)[:5]
unbeschriftete_globalindizes = np.flatnonzero(unbeschriftet)
unsicherste_global = unbeschriftete_globalindizes[unsicherste_lokal]

unsicherheitsbericht = pd.DataFrame(
    {
        "Index": unsicherste_global,
        "Vorhersage": semi_modell.transduction_[unsicherste_global],
        "Sicherheit": sicherheit[unsicherste_lokal],
        "Wahre_Klasse_nur_Kontrolle": y_blobs[unsicherste_global],
    }
)
display(unsicherheitsbericht.round(3))

> **Musterantwort und Interpretation**
>
> LabelSpreading übernimmt Annahmen über lokale Ähnlichkeit und kann Fehler der wenigen Startlabels verbreiten. Die erzeugten Labels sind Modellvorhersagen und keine neue Beobachtung der Wahrheit. Unsicherheiten sollten gespeichert und besonders unsichere Fälle bevorzugt manuell geprüft werden.

### Aufgabe 5: Count-, TF-IDF- und Hashing-Merkmale untersuchen

Schreiben Sie eine einfache Bereinigungsfunktion für deutsche Texte, die Kleinschreibung anwendet, technische Ziffern entfernt, Satzzeichen durch Leerzeichen ersetzt und Mehrfachleerzeichen reduziert.

Erzeugen Sie anschließend für die bereinigten Texte:

1. Count-Merkmale mit Unigrammen und Bigrammen,
2. TF-IDF-Merkmale mit denselben n-Grammen,
3. Hashing-Merkmale mit 64 Dimensionen.

Geben Sie Form, Typ und Dichte der drei Matrizen aus. Zeigen Sie außerdem die zehn häufigsten Count-Merkmale.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def bereinige_text(text):
    """Vereinheitlicht technische Oberflächenmerkmale, ohne Wörter zu erfinden."""
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

bereinigte_texte = text_daten["Text"].map(bereinige_text)

count_vectorizer = CountVectorizer(ngram_range=(1, 2), min_df=2)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
hashing_vectorizer = HashingVectorizer(
    n_features=64,
    alternate_sign=False,
    norm=None,
    ngram_range=(1, 2),
)

X_count = count_vectorizer.fit_transform(bereinigte_texte)
X_tfidf = tfidf_vectorizer.fit_transform(bereinigte_texte)
X_hash = hashing_vectorizer.transform(bereinigte_texte)

for name, matrix in {"Count": X_count, "TF-IDF": X_tfidf, "Hashing": X_hash}.items():
    dichte = matrix.nnz / (matrix.shape[0] * matrix.shape[1])
    print(f"{name}: Form={matrix.shape}, Typ={type(matrix).__name__}, Dichte={dichte:.4f}")

haeufigkeiten = np.asarray(X_count.sum(axis=0)).ravel()
merkmale = count_vectorizer.get_feature_names_out()
top_indizes = np.argsort(haeufigkeiten)[-10:][::-1]
top_woerter = pd.DataFrame(
    {"Merkmal": merkmale[top_indizes], "Anzahl": haeufigkeiten[top_indizes]}
)
display(top_woerter)

> **Musterantwort und Interpretation**
>
> Hashing benötigt kein gelerntes Vokabular und ist speichergünstig für Datenströme. Die Spalten lassen sich jedoch nicht zuverlässig in ursprüngliche Wörter zurückübersetzen, und verschiedene Begriffe können in derselben Hash-Spalte kollidieren. Dadurch wird die Interpretation schwieriger.

### Aufgabe 6: Integrationsaufgabe: Textpipelines vergleichen und Fehler analysieren

Teilen Sie die Textdaten stratifiziert in Training und Test. Vergleichen Sie drei vollständige Pipelines:

- CountVectorizer plus MultinomialNB,
- TfidfVectorizer plus LogisticRegression,
- TfidfVectorizer plus LinearSVC.

Nutzen Sie dieselbe Bereinigungsfunktion als `preprocessor`. Berechnen Sie Accuracy und Macro-F1. Wählen Sie nach Macro-F1 das beste Modell, erstellen Sie eine Konfusionsmatrix und geben Sie alle falsch klassifizierten Testtexte mit wahrer und vorhergesagter Klasse aus.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay

# ============================================================
# MUSTERLÖSUNG
# ============================================================

text_train, text_test, label_train, label_test = train_test_split(
    text_daten["Text"],
    text_daten["Kategorie"],
    test_size=0.30,
    stratify=text_daten["Kategorie"],
    random_state=RANDOM_SEED,
)

text_modelle = {
    "Count + NB": Pipeline(
        [
            ("vektor", CountVectorizer(preprocessor=bereinige_text, ngram_range=(1, 2), min_df=1)),
            ("modell", MultinomialNB(alpha=1.0)),
        ]
    ),
    "TF-IDF + LogReg": Pipeline(
        [
            ("vektor", TfidfVectorizer(preprocessor=bereinige_text, ngram_range=(1, 2), min_df=1)),
            ("modell", LogisticRegression(max_iter=1500, random_state=RANDOM_SEED)),
        ]
    ),
    "TF-IDF + LinearSVC": Pipeline(
        [
            ("vektor", TfidfVectorizer(preprocessor=bereinige_text, ngram_range=(1, 2), min_df=1)),
            ("modell", LinearSVC(random_state=RANDOM_SEED)),
        ]
    ),
}

text_berichte = []
text_vorhersagen = {}
for name, text_modell in text_modelle.items():
    text_modell.fit(text_train, label_train)
    prognose = text_modell.predict(text_test)
    text_vorhersagen[name] = prognose
    text_berichte.append(
        {
            "Modell": name,
            "Accuracy": accuracy_score(label_test, prognose),
            "Macro_F1": f1_score(label_test, prognose, average="macro"),
        }
    )

text_vergleich = pd.DataFrame(text_berichte).sort_values("Macro_F1", ascending=False)
display(text_vergleich.round(3))

bester_name = text_vergleich.iloc[0]["Modell"]
bestes_textmodell = text_modelle[bester_name]
beste_prognose = text_vorhersagen[bester_name]
print("Gewähltes Modell:", bester_name)

ConfusionMatrixDisplay.from_predictions(label_test, beste_prognose)
plt.title("Konfusionsmatrix des gewählten Textmodells")
plt.show()

fehler_maske = np.asarray(label_test) != beste_prognose
fehlerbericht = pd.DataFrame(
    {
        "Text": np.asarray(text_test)[fehler_maske],
        "Wahr": np.asarray(label_test)[fehler_maske],
        "Vorhergesagt": beste_prognose[fehler_maske],
    }
)
print("Falsch klassifizierte Testtexte:")
display(fehlerbericht)

> **Musterantwort und Interpretation**
>
> Accuracy zählt alle richtigen Vorhersagen gemeinsam und kann von häufigen Klassen dominiert werden. Macro-F1 berechnet F1 pro Klasse und mittelt die Klassen gleichgewichtet. Damit werden Schwächen in kleineren oder schwierigeren Kategorien deutlicher. Bei sehr kleinen Testsätzen bleibt die Unsicherheit beider Kennzahlen hoch, weshalb einzelne Fehler ebenfalls gelesen werden sollten.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?